# Electricity balances

This notebook analyzes a previously assembled **multi-scenario PEXL project** with hourly timestep data.

Create the project first with `howto_create_project_from_different_timeseries_files.ipynb`. This keeps file collection and project assembly separate from the actual scenario analysis.




## Load the prepared project

The saved project already contains the scenarios and their timestep data, so the source Excel files do not need to be reopened here.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pickle
import pandas as pd
import plotly.io as pio

pio.renderers.default = "notebook"
print(pio.renderers.default)


project_path = Path(
    r"C:\Users\schneids\Nextcloud\EE\6_Daten\Quartiere"
    r"\1220_Wien_Aspern_Seestadt\Unterlagen PV Überschüsse"
    r"\PEexcel Auswertung\aspern_pvcheck_project.pkl"
)

with project_path.open("rb") as file:
    project = pickle.load(file)

project.scenario_names()



In [ ]:
project.column_names()

In [ ]:
from pexl.schema.current import TIMESERIES_META
TIMESERIES_META.__dict__

In [ ]:
project.scenarios[1].timeseries.columns

In [ ]:
util = ["Monat_nr", "hour_of_the_day"]

negative= [
        "E_plugAuxLight",
        "Ev_min",
        "Edhw_min",
        "Edhw1_flex_final",
        "Edhw2_flex_final",
        "Ec_min",
        "Ec_flex_1el_final",
        "Ec_flex_3el_final",
        "Eh_min",
        "Eh_flex_1el_final",
        "Eh_flex_3el_final",
        "Eev_min",
        "PV_to_Eev_flex"
    ]
positive= [
    "E_grid",
    #"PV_to_user",
    "PV_total_direct_use",
    "PV_total_flex_use",
    "PV_to_Egrid",
]

In [ ]:
SUPERBLOCKS = {
   #"Superblock F5+F6+F7": ["F5", "F6", "F7"],
   # "Superblock F9B+F10+F12+F13": ["F9B", "F10", "F12", "F13"],
    "Seecarrée": ["F3", "F5", "F6", "F7", "F9B", "F10", "F12", "F13", "E7A"],
    "Seeterrassen": ["G11", "H1", "H5"],
    "Seestadt Süd": ["J6B", "C1A", "D18C", "C4 BT4+5"],
    "Gesamt": project.scenario_names()
}

from pexl.model.scenario import Scenario


for block_name, scenario_names in SUPERBLOCKS.items():
    scenarios = [project["Aspern PVCheck _ "+name] for name in scenario_names]

    nfa_total = sum(s.v.NFA_total for s in scenarios)
    

    data = {}

    for var in negative + positive:
        data[var] = sum(
            s.timeseries[var] * s.v.NFA_total
            for s in scenarios
        ) / nfa_total

    for var in util:
        data[var] = scenarios[0].timeseries[var]

    block = Scenario(block_name)
    block.v.project_scenario_name = block_name
    block.set_timeseries(pd.DataFrame(data))
    block.v.GFA_total = sum(s.v.GFA_total for s in scenarios)

    project.add_scenario(block)

In [ ]:
from pexl.analysis.temporal import SEASONS, add_season
from pexl.plot.styles import color_for
import pexl.plot

for scenario in project:
    ts = add_season(scenario.timeseries.to_frame())

    seasonal = (
        ts
        .groupby(["season", "hour_of_the_day"], observed=True)[
            positive + negative
        ]
        .mean()
        .reset_index()
    )

    season_order = [
        season
        for season, _, _ in SEASONS
    ]

    season_labels = {
        season: label.de
        for season, label, _ in SEASONS
    }

    fig = pexl.plot.electricity_balance_facets(
        seasonal,
        positive=positive,
        negative=negative,
        facet="season",
        facet_order=season_order,
        facet_labels=season_labels,
        x="hour_of_the_day",
        title=scenario.name,
        x_title="Tagesstunde",
        y_title="Mittlere Leistung [W/m²]",
        omit_empty=True
    )
    fig.update_xaxes(
        title_text="Tagesstunde",
        tickmode="array",
        tickvals=[0, 6, 12, 18],
        ticktext=["0", "6", "12", "18"],
    )
    fig.show()

### HTML VIEW of results
To export as html, *save workbook* , specify filename and run cell below

In [ ]:
#!jupyter nbconvert --to html --no-input --HTMLExporter.exclude_output_prompt=True howto_analyze_electricity_balance.ipynb